In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")          # one BLAS thread per worker process
import argparse, sys, time, warnings
import numpy as np
import pandas as pd
from scipy import stats, optimize, linalg
from scipy.spatial.distance import cdist
from concurrent.futures import ProcessPoolExecutor

warnings.filterwarnings("ignore")
U0 = 10.0                      # known threshold used in the simulation (same at all sites)
LAM0 = 2.5                     # Poisson peaks per year (3.20.2)
Z90 = stats.norm.ppf(0.95)     # 1.645 -> central 90% intervals
B0, B1, B2 = 1.8, 0.5, -0.4    # trend coefficients of ln sigma_u(s)  (3.20.1)
ALL = ["EVT-NN", "EVT-IDW", "OK", "EVT-SK (direct)", "EVT-SK (composed)",
       "Bayes EVT-SK (direct)", "Bayes EVT-SK (composed)"]
PROP = ALL[3:]
CONV = ALL[:3]
LOG = []


def say(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    LOG.append(s)

In [ ]:
# 1. Covariance models, covariates, return-level maps
# =====================================================================================
def cov_fun(h, s2, phi, fam):
    """Covariance (3.10.4)-(3.10.7), x = |h|/phi."""
    x = np.asarray(h, float) / phi
    if fam == "exp":
        r = np.exp(-x)
    elif fam == "m32":
        r = (1 + x) * np.exp(-x)
    elif fam == "m52":
        r = (1 + x + x ** 2 / 3) * np.exp(-x)
    elif fam == "gauss":
        r = np.exp(-x ** 2)
    else:
        raise ValueError(fam)
    return s2 * r


def covariates(X):
    """Smooth elevation (E) and rainfall (P) indices used in the simulation (3.3, 3.20.1)."""
    x, y = X[:, 0] / 100.0, X[:, 1] / 100.0
    return 1.6 * (x - .5) + .5 * np.sin(2 * np.pi * y), 1.4 * (y - .5) - .5 * np.cos(2 * np.pi * x)


def design(X, trend=True):
    if not trend:
        return np.ones((len(X), 1))
    E, P = covariates(X)
    return np.c_[np.ones(len(X)), E, P]


def rl(sig, xi, lam, T, u=U0):
    """GPD return level (3.8.1); Gumbel limit when xi -> 0. Fully vectorised."""
    sig, xi, lam = np.broadcast_arrays(np.asarray(sig, float), np.asarray(xi, float), np.asarray(lam, float))
    small = np.abs(xi) < 1e-6
    xs = np.where(small, 1.0, xi)
    return u + np.where(small, sig * np.log(lam * T), sig * ((lam * T) ** xs - 1) / xs)


def Psi(th3, T, u=U0):
    """Composed return-level map (3.12.1); th3[...] = (ln sigma, xi, ln lambda)."""
    xi = np.clip(th3[..., 1], -0.4, 0.9)
    return rl(np.exp(th3[..., 0]), xi, np.exp(th3[..., 2]), T, u)


def Psi_grad(th3, T):
    """Gradient (3.14.3) wrt (ln sigma, xi, ln lambda)."""
    sig = np.exp(th3[..., 0]); xi = np.clip(th3[..., 1], -0.4, 0.9); lam = np.exp(th3[..., 2])
    xs = np.where(np.abs(xi) < 1e-4, 1e-4, xi)
    a = (lam * T) ** xs
    return np.stack([sig * (a - 1) / xs, sig / xs * (a * np.log(lam * T) - (a - 1) / xs), sig * a], -1)

In [ ]:
# =====================================================================================
# 2. At-site EVT (3.5-3.9)
# =====================================================================================
def gpd_nll(par, x):
    ls, xi = par
    if xi < -0.45 or xi > 1.0:
        return 1e10
    if abs(xi) < 1e-8:
        return len(x) * ls + x.sum() / np.exp(ls)
    z = 1 + xi * x / np.exp(ls)
    if np.any(z <= 0):
        return 1e10
    return len(x) * ls + (1 + 1 / xi) * np.log(z).sum()          # -l(theta), (3.7.2)


def fit_gpd(x):
    """Numerical ML (3.7.4-3.7.5 solved by Nelder-Mead on (ln sigma, xi)); returns (ln sigma, xi)."""
    x = np.asarray(x, float); N = len(x); m = x.mean()
    v = x.var(ddof=1) if N > 1 else m * m
    xi0 = float(np.clip(0.5 * (1 - m * m / max(v, 1e-12)), -0.3, 0.6)); s0 = max(m * (1 - xi0), 1e-6)
    if xi0 < 0 and 1 + xi0 * x.max() / s0 <= 0:
        xi0, s0 = 0.05, m
    r = optimize.minimize(gpd_nll, [np.log(s0), xi0], args=(x,), method="Nelder-Mead",
                          options=dict(xatol=1e-5, fatol=1e-9, maxiter=500))
    return float(r.x[0]), float(r.x[1])


def gpd_cov(sig, xi, N):
    """Asymptotic covariance of (sigma_hat, xi_hat), (3.7.9)."""
    return (1 + xi) / N * np.array([[2 * sig ** 2 * (1 + xi), -sig], [-sig, 1 + xi]])


def rl_logvar(sig, xi, lam, ny, N, T):
    """Delta-method variance of ln z_T, (3.9.1)-(3.9.3)."""
    xi = float(np.clip(xi, -0.4, 0.9)); xs = xi if abs(xi) > 1e-4 else 1e-4
    a = (lam * T) ** xs
    g = np.array([(a - 1) / xs, sig / xs * (a * np.log(lam * T) - (a - 1) / xs)])
    v = g @ gpd_cov(sig, max(xi, -0.25), N) @ g + (sig * a / lam) ** 2 * lam / ny
    z = rl(sig, xi, lam, T)
    return float(v / z ** 2)


def boot_logvar(x, ny, rng, B, T):
    """Non-parametric bootstrap variance of ln z_T for short records (3.9)."""
    N = len(x); vals = []
    for _ in range(B):
        ls, xi = fit_gpd(rng.choice(x, N, replace=True))
        vals.append(np.log(rl(np.exp(ls), xi, max(rng.poisson(N), 1) / ny, T)))
    vals = np.array(vals)[np.isfinite(vals)]
    return float(np.var(vals, ddof=1)) if len(vals) > 5 else 2.0


def bayes_site(x, ny, rng, m_sig, s_sig=1.0, B=300, ng=50):
    """Bayesian at-site GPD posterior (3.12.3) on a 2-D grid, priors ln sigma ~ N(m_sig, s_sig^2),
    xi ~ N(0, 0.3^2).  lambda | data ~ Gamma(N + 1/2, ny) (Poisson-Jeffreys).  Returns B draws."""
    N = len(x); ls_h, xi_h = fit_gpd(x)
    se_l = np.sqrt(2) * (1 + np.clip(xi_h, -.4, .9)) / np.sqrt(N)
    se_x = (1 + np.clip(xi_h, -.4, .9)) / np.sqrt(N)
    lg = np.linspace(ls_h - 6 * se_l, ls_h + 6 * se_l, ng)
    xg = np.linspace(max(-0.45, xi_h - 6 * se_x), min(1.0, xi_h + 6 * se_x), ng)
    LS, XI = np.meshgrid(lg, xg, indexing="ij")
    XS = np.where(np.abs(XI) < 1e-6, 1e-6, XI)
    z = 1 + XS[..., None] * x / np.exp(LS)[..., None]
    with np.errstate(invalid="ignore", divide="ignore"):
        ll = -N * LS - (1 + 1 / XS) * np.where(z > 0, np.log(np.where(z > 0, z, 1)), 0).sum(-1)
    ll = np.where((z > 0).all(-1), ll, -np.inf)
    lp = ll + stats.norm.logpdf(LS, m_sig, s_sig) + stats.norm.logpdf(XI, 0, 0.3)
    w = np.exp(lp - lp.max()).ravel(); w /= w.sum()
    idx = rng.choice(w.size, size=B, p=w); i, j = np.unravel_index(idx, LS.shape)
    dl = lg[1] - lg[0]; dx = xg[1] - xg[0] if ng > 1 else 0
    ls = LS[i, j] + (rng.random(B) - .5) * dl
    xi = np.clip(XI[i, j] + (rng.random(B) - .5) * dx, -0.45, 1.0)
    lam = rng.gamma(N + 0.5, 1.0, B) / ny
    return dict(ls=ls, xi=xi, lam=lam, mle=(ls_h, xi_h))


def atsite(data, rng, cfg, bayes=True):
    """Stage 1 + 2 of EVT-SK: fit every gauge and convert estimation uncertainty to noise tau_i^2."""
    n = len(data["exc"]); T = data["T"]
    th = np.zeros(n); tau2 = np.zeros(n); ing = np.zeros((n, 3)); ingv = np.zeros((n, 3))
    for i, x in enumerate(data["exc"]):
        N = len(x); ny = data["ny"][i]; ls, xi = fit_gpd(x); lam = N / ny; sig = np.exp(ls)
        xc = np.clip(xi, -0.25, 0.9)
        th[i] = np.log(rl(sig, xi, lam, T)); ing[i] = [ls, xi, np.log(lam)]
        ingv[i] = [2 * (1 + xc) ** 2 / N, (1 + xc) ** 2 / N, 1.0 / N]     # (3.7.10)-(3.7.11)
        tau2[i] = (boot_logvar(x, ny, rng, cfg["B_boot"], T) if N < cfg["boot_min"]
                   else rl_logvar(sig, xi, lam, ny, N, T))
    out = dict(th=th, tau2=np.clip(np.nan_to_num(tau2, nan=2.0), 1e-4, 2.0), ing=ing, ingv=ingv)
    if bayes:
        msig = float(np.mean([np.log(x.mean()) for x in data["exc"]]))      # empirical-Bayes prior mean
        thb = np.zeros(n); tb = np.zeros(n); ib = np.zeros((n, 3)); ivb = np.zeros((n, 3)); draws = []
        for i, x in enumerate(data["exc"]):
            d = bayes_site(x, data["ny"][i], rng, msig, 1.0, cfg["B_site"])
            lz = np.log(rl(np.exp(d["ls"]), d["xi"], d["lam"], T))
            thb[i] = np.median(lz); tb[i] = lz.var(ddof=1)                    # (3.12.4)
            lamL = np.log(d["lam"])
            ib[i] = [np.median(d["ls"]), np.median(d["xi"]), np.median(lamL)]
            ivb[i] = [d["ls"].var(ddof=1), d["xi"].var(ddof=1), lamL.var(ddof=1)]
            draws.append(d)
        out.update(th_b=thb, tau2_b=np.minimum(tb, 2.0), ing_b=ib, ingv_b=ivb, draws=draws)
    return out

In [ ]:
# =====================================================================================
# 3. Spatial stage: REML, Stochastic Kriging BLUP, variance split (3.10-3.11)
# =====================================================================================
LB = np.log([1e-5, 1.5, 1e-7]); UB = np.log([50., 400., 5.])


def _V(D, par, fam, delta):
    V = cov_fun(D, par[0], par[1], fam)
    V[np.diag_indices_from(V)] += par[2] + delta + 1e-8
    return V


def reml_nll(lp, th, F, D, delta, fam):
    """Negative restricted log-likelihood (3.10.9)."""
    try:
        L = linalg.cho_factor(_V(D, np.exp(lp), fam, delta), lower=True)
        A = F.T @ linalg.cho_solve(L, F)
        beta = np.linalg.solve(A, F.T @ linalg.cho_solve(L, th))
        r = th - F @ beta
        return (np.log(np.diag(L[0])).sum() + 0.5 * np.linalg.slogdet(A)[1]
                + 0.5 * r @ linalg.cho_solve(L, r))
    except Exception:
        return 1e10


def fit_reml(th, F, D, delta, fam):
    best = None; v0 = max(np.var(th), 1e-3)
    for phi0 in (15., 50.):
        r = optimize.minimize(reml_nll, np.log([.5 * v0, phi0, .05 * v0]), args=(th, F, D, delta, fam),
                              method="L-BFGS-B", bounds=list(zip(LB, UB)))
        if best is None or r.fun < best.fun:
            best = r
    return best.x


def sk_predict(th, F, D, delta, par, fam, Xg, X0, F0, want_w=True):
    """GLS trend (3.11.3), BLUP (3.11.4), kriging variance (3.11.5), at-site split (3.11.8)."""
    L = linalg.cho_factor(_V(D, par, fam, delta), lower=True)
    Sig0 = cov_fun(cdist(Xg, X0), par[0], par[1], fam)
    ViS = linalg.cho_solve(L, Sig0); ViF = linalg.cho_solve(L, F); A = F.T @ ViF
    beta = np.linalg.solve(A, F.T @ linalg.cho_solve(L, th))
    mean = F0 @ beta + Sig0.T @ linalg.cho_solve(L, th - F @ beta)
    R = F0.T - F.T @ ViS; AiR = np.linalg.solve(A, R)
    var = np.maximum(par[0] - np.sum(Sig0 * ViS, 0) + np.sum(R * AiR, 0), 1e-12)
    if not want_w:
        return mean, var, None
    W = ViS + ViF @ AiR                                              # weights (3.11.7)
    return mean, var, (W ** 2 * delta[:, None]).sum(0)


def krige_cols(cols, vars_, F, D, Xg, X0, F0, fam, scale):
    """Frequentist SK, each column kriged separately (cross-covariances = 0)."""
    k = cols.shape[1]; m = np.zeros((len(X0), k)); v = m.copy(); a = m.copy(); pars = []
    for j in range(k):
        delta = scale * vars_[:, j]
        par = np.exp(fit_reml(cols[:, j], F, D, delta, fam)); pars.append(par)
        m[:, j], v[:, j], a[:, j] = sk_predict(cols[:, j], F, D, delta, par, fam, Xg, X0, F0)
    return m, v, a, pars


def eta_posterior(th, F, D, delta, fam, B, rng):
    """Importance-sampling posterior of eta=(sigma_M^2, phi, tau0^2), (3.12.5).
    Hyperpriors (log-normal): sigma_M^2 ~ LN(ln 0.1, 1.5^2); phi ~ LN(ln 30, 0.8^2); tau0^2 ~ LN(ln 0.02, 2^2).
    Proposal: Gaussian on the log scale centred at the REML mode."""
    c = fit_reml(th, F, D, delta, fam); s = np.array([.6, .5, 1.0]); Bp = max(2 * B, 200)
    prop = c + rng.standard_normal((Bp, 3)) * s
    lw = np.full(Bp, -np.inf)
    for b in range(Bp):
        if np.all(prop[b] >= LB) and np.all(prop[b] <= UB):
            lpr = (stats.norm.logpdf(prop[b, 0], np.log(.1), 1.5) + stats.norm.logpdf(prop[b, 1], np.log(30), .8)
                   + stats.norm.logpdf(prop[b, 2], np.log(.02), 2.0))
            lw[b] = -reml_nll(prop[b], th, F, D, delta, fam) + lpr - stats.norm.logpdf(prop[b], c, s).sum()
    w = np.exp(lw - lw.max()); w /= w.sum()
    return np.exp(prop[rng.choice(Bp, size=B, p=w)])


def krige_cols_bayes(cols, vars_, F, D, Xg, X0, F0, fam, scale, B, rng):
    """Bayesian SK: predictive draws mixing over p(eta | data), (3.12.6)."""
    k = cols.shape[1]; dr = np.zeros((B, len(X0), k))
    for j in range(k):
        delta = scale * vars_[:, j]
        etas = eta_posterior(cols[:, j], F, D, delta, fam, B, rng)
        for b in range(B):
            m, v, _ = sk_predict(cols[:, j], F, D, delta, etas[b], fam, Xg, X0, F0, want_w=False)
            dr[b, :, j] = m + np.sqrt(v) * rng.standard_normal(len(X0))
    return dr

In [ ]:
# =====================================================================================
# 4. Data-generating process (3.20.1-3.20.3)
# =====================================================================================
BASE = dict(n=None, ny=(10, 50), xi=0.15, xi_sd=0.07, phi=30.0, sigM=0.35, cov="m32",
            contam_g=0.0, contam_x=0.15)
SCENARIOS = {
    "S1 Baseline": {}, "S2 Short records": dict(ny=(5, 15)), "S3 Heavy tail": dict(xi=0.35),
    "S4 Few gauges": dict(n=14), "S5 Contamination": dict(contam_g=0.2),
    "S6 Short range": dict(phi=8.0), "S7 Unequal records": dict(ny=(5, 80)),
    "S8 Gaussian cov": dict(cov="gauss")}
CFG = dict(fam_fit="m32", T=100, noise_scale=1.0, use_trend=True, B_site=300, B=150,
           B_boot=50, boot_min=20, nval=40)


def simulate(rng, sc, T=100, nval=40):
    s = {**BASE, **sc}
    n = s["n"] or int(rng.integers(29, 31))
    Xg = rng.uniform(0, 100, (n, 2)); Xv = rng.uniform(0, 100, (nval, 2)); Xa = np.vstack([Xg, Xv])
    D = cdist(Xa, Xa); N_ = len(Xa)
    M = np.linalg.cholesky(cov_fun(D, s["sigM"] ** 2, s["phi"], s["cov"]) + 1e-8 * np.eye(N_)) @ rng.standard_normal(N_)
    xi = s["xi"] + s["xi_sd"] * np.linalg.cholesky(cov_fun(D, 1., 40., "m32") + 1e-8 * np.eye(N_)) @ rng.standard_normal(N_)
    xi = np.clip(xi, -0.2, 0.7)
    E, P = covariates(Xa)
    sig = np.exp(B0 + B1 * E + B2 * P + M)
    th = np.log(rl(sig, xi, LAM0, T))                                     # (3.20.3)
    ny = rng.integers(s["ny"][0], s["ny"][1] + 1, n)
    bad = set(rng.choice(n, int(round(s["contam_g"] * n)), replace=False).tolist()) if s["contam_g"] > 0 else set()
    exc = []
    for i in range(n):
        N = max(int(rng.poisson(LAM0 * ny[i])), 8)
        x = stats.genpareto.rvs(c=xi[i], scale=sig[i], size=N, random_state=rng)   # (3.20.2)
        if i in bad:                                                       # rating-curve errors
            x = np.where(rng.random(N) < s["contam_x"], x * rng.uniform(2, 4, N), x)
        exc.append(x)
    return dict(Xg=Xg, Xv=Xv, ny=ny, exc=exc, th_g=th[:n], th_v=th[n:], T=T)


In [ ]:
# =====================================================================================
# 5. The seven predictors and their scores (3.18, 3.20.4)
# =====================================================================================
def predict_methods(data, rng, cfg, X0, methods=ALL):
    T = data["T"]; Xg = data["Xg"]; fam = cfg["fam_fit"]; sc = cfg["noise_scale"]
    Fg = design(Xg, cfg["use_trend"]); F0 = design(X0, cfg["use_trend"]); D = cdist(Xg, Xg); D0 = cdist(X0, Xg)
    need_b = any(m.startswith("Bayes") for m in methods)
    a = atsite(data, rng, cfg, bayes=need_b); out = {}
    if "EVT-NN" in methods:
        i = D0.argmin(1); out["EVT-NN"] = dict(mu=a["th"][i], sd=np.sqrt(a["tau2"][i]))
    if "EVT-IDW" in methods:
        w = np.maximum(D0, 1e-9) ** -2.0; w /= w.sum(1, keepdims=True)
        out["EVT-IDW"] = dict(mu=w @ a["th"], sd=w @ np.sqrt(a["tau2"]))
    if "OK" in methods:                                                    # Delta = 0
        z0 = np.zeros(len(Xg)); par = np.exp(fit_reml(a["th"], Fg, D, z0, fam))
        m, v, _ = sk_predict(a["th"], Fg, D, z0, par, fam, Xg, X0, F0, False)
        out["OK"] = dict(mu=m, sd=np.sqrt(v), par=par)
    if "EVT-SK (direct)" in methods:                                       # (3.12.2)
        delta = sc * a["tau2"]; par = np.exp(fit_reml(a["th"], Fg, D, delta, fam))
        m, v, _ = sk_predict(a["th"], Fg, D, delta, par, fam, Xg, X0, F0, False)
        out["EVT-SK (direct)"] = dict(mu=m, sd=np.sqrt(v), par=par)
    if "EVT-SK (composed)" in methods:                                     # (3.12.1), (3.14.4)
        m, v, at, _ = krige_cols(a["ing"], a["ingv"], Fg, D, Xg, X0, F0, fam, sc)
        z = Psi(m, T); G = Psi_grad(m, T)
        out["EVT-SK (composed)"] = dict(mu=np.log(z), sd=np.sqrt((G ** 2 * v).sum(1)) / z)
    if "Bayes EVT-SK (direct)" in methods:
        dr = krige_cols_bayes(a["th_b"][:, None], a["tau2_b"][:, None], Fg, D, Xg, X0, F0, fam, sc, cfg["B"], rng)
        out["Bayes EVT-SK (direct)"] = dict(draws=dr[..., 0])
    if "Bayes EVT-SK (composed)" in methods:
        dr = krige_cols_bayes(a["ing_b"], a["ingv_b"], Fg, D, Xg, X0, F0, fam, sc, cfg["B"], rng)
        out["Bayes EVT-SK (composed)"] = dict(draws=np.log(Psi(dr, T)))
    out["_atsite"] = a
    return out


def crps_gauss(y, m, s):                                                   # (3.18.7)
    w = (y - m) / s
    return s * (w * (2 * stats.norm.cdf(w) - 1) + 2 * stats.norm.pdf(w) - 1 / np.sqrt(np.pi))


def crps_draws(dr, y):                                                     # (3.18.8)
    B = dr.shape[0]; srt = np.sort(dr, 0); k = np.arange(1, B + 1)[:, None]
    return np.abs(dr - y).mean(0) - ((2 * k - B - 1) * srt).sum(0) / B ** 2


def score(p, truth):
    if "draws" in p:
        dr = p["draws"]; mu = np.median(dr, 0); lo, hi = np.quantile(dr, [.05, .95], 0)
        crps = crps_draws(dr, truth); pit = (dr <= truth).mean(0)
    else:
        mu, sd = p["mu"], np.maximum(p["sd"], 1e-4); lo, hi = mu - Z90 * sd, mu + Z90 * sd
        crps = crps_gauss(truth, mu, sd); pit = stats.norm.cdf((truth - mu) / sd)
    return (dict(rmse=float(np.sqrt(np.mean((mu - truth) ** 2))), bias=float(np.mean(mu - truth)),
                 cov=float(np.mean((truth >= lo) & (truth <= hi))), width=float(np.mean(hi - lo)),
                 crps=float(np.mean(crps))), pit)


def run_rep(args):
    name, sc, rep, seed, cfg, methods, tag = args
    try:
        rng = np.random.default_rng(seed)
        data = simulate(rng, sc, cfg["T"], cfg["nval"])
        pred = predict_methods(data, rng, cfg, data["Xv"], methods)
        rows = []
        for m in methods:
            s, _ = score(pred[m], data["th_v"])
            rows.append(dict(tag=tag, scenario=name, rep=rep, method=m, **s))
        return rows
    except Exception as e:                                                  # keep the MC running
        return [dict(tag=tag, scenario=name, rep=rep, method="ERROR", err=repr(e))]


def run_tasks(tasks, jobs, label):
    t0 = time.time(); res = []; n = len(tasks)
    if jobs <= 1:
        for k, t in enumerate(tasks):
            res += run_rep(t)
            if (k + 1) % max(1, n // 10) == 0:
                say(f"  [{label}] {k + 1}/{n} done ({time.time() - t0:.0f}s)")
    else:
        with ProcessPoolExecutor(max_workers=jobs) as ex:
            for k, r in enumerate(ex.map(run_rep, tasks, chunksize=2)):
                res += r
                if (k + 1) % max(1, n // 10) == 0:
                    say(f"  [{label}] {k + 1}/{n} done ({time.time() - t0:.0f}s)")
    df = pd.DataFrame(res)
    if "err" in df and (df.method == "ERROR").any():
        say(f"  WARNING: {(df.method == 'ERROR').sum()} failed replicate(s) in {label}; e.g. {df.err.dropna().iloc[0]}")
        df = df[df.method != "ERROR"]
    return df.drop(columns=[c for c in ["err"] if c in df])


In [ ]:
# =====================================================================================
# 6. Hypothesis tests (3.20.5)
# =====================================================================================
def holm(p):
    """Holm (1979) adjusted p-values (3.20.11)."""
    p = np.asarray(p, float); m = len(p); adj = np.empty(m); run = 0.0
    for j, idx in enumerate(np.argsort(p)):
        run = max(run, (m - j) * p[idx]); adj[idx] = min(1.0, run)
    return adj


def paired_tests(df):
    piv = df.pivot_table(index=["scenario", "rep"], columns="method", values="rmse")
    rows = []
    for sc in df.scenario.unique():
        sub = piv.loc[sc]
        for pm in PROP:
            for cm in CONV:
                d = (sub[cm] - sub[pm]).dropna()
                try:
                    p = stats.wilcoxon(d, alternative="greater").pvalue          # (3.20.10)
                except Exception:
                    p = 1.0
                rows.append(dict(scenario=sc, proposed=pm, conventional=cm, R=len(d),
                                 mean_RMSE_prop=sub[pm].mean(), mean_RMSE_conv=sub[cm].mean(),
                                 reduction_pct=float((100 * (1 - sub[pm] / sub[cm])).mean()),   # (3.18.10)
                                 win_rate=float((sub[pm] < sub[cm]).mean()), p_raw=p))
    t = pd.DataFrame(rows)
    h2 = t[t.conventional == "EVT-NN"].copy(); h2["p_holm"] = holm(h2.p_raw); h2["reject_H0"] = h2.p_holm < .05
    h3 = t.copy(); h3["p_holm"] = holm(h3.p_raw); h3["reject_H0"] = h3.p_holm < .05
    return h2, h3


In [ ]:
# =====================================================================================
# 7. Goodness of fit: Cramer-von Mises with parametric bootstrap (Hypothesis 1)
# =====================================================================================
def cvm_stat(x, ls, xi):
    x = np.sort(x); N = len(x); s = np.exp(ls)
    H = 1 - np.maximum(1 + xi * x / s, 0) ** (-1 / xi) if abs(xi) > 1e-8 else 1 - np.exp(-x / s)
    return 1 / (12 * N) + np.sum((H - (2 * np.arange(1, N + 1) - 1) / (2 * N)) ** 2)      # (3.20.7)


def cvm_test(x, B, rng):
    ls, xi = fit_gpd(x); W = cvm_stat(x, ls, xi); N = len(x); cnt = 0
    for _ in range(B):
        xb = stats.genpareto.rvs(c=xi, scale=np.exp(ls), size=N, random_state=rng)
        lb, xb_i = fit_gpd(xb); cnt += cvm_stat(xb, lb, xb_i) >= W
    return W, (1 + cnt) / (B + 1)                                                          # (3.20.8)


def stage_gof(out, seed, B):
    rng = np.random.default_rng(seed)
    data = simulate(rng, SCENARIOS["S1 Baseline"], 100, 40)
    rows = []
    for i, x in enumerate(data["exc"]):
        W, p = cvm_test(x, B, rng); rows.append(dict(gauge=i + 1, N=len(x), years=data["ny"][i], W2=W, p=p))
    t = pd.DataFrame(rows); K = int((t.p > .05).sum()); n = len(t)
    pb = float(stats.binom.sf(K - 1, n, .5))                                               # (3.20.9)
    t.to_csv(f"{out}/tables/H1_cvm_gof_by_gauge.csv", index=False)
    say(f"\n[H1] CvM bootstrap GOF: {K}/{n} gauges adequate (p>0.05); rejections={n - K} "
        f"(~{0.05 * n:.1f} expected by chance); binomial p={pb:.3g} -> "
        f"{'REJECT H0 (GPD adequate)' if pb < .05 else 'do not reject H0'}")
    pd.DataFrame([dict(n=n, K_adequate=K, p_binom=pb, reject_H0=pb < .05)]).to_csv(
        f"{out}/tables/H1_binomial_test.csv", index=False)
    return t



In [ ]:
# =====================================================================================
# 8. Cross-validation (3.16-3.18)
# =====================================================================================
def cv_metrics(th, pred, var, tau2, th_true=None):
    e = pred - th
    out = dict(RMSE=np.sqrt(np.mean(e ** 2)), MAE=np.mean(np.abs(e)),
               NSE=1 - np.sum(e ** 2) / np.sum((th - th.mean()) ** 2),
               Coverage90=np.mean(np.abs(e) <= Z90 * np.sqrt(var + tau2)),                 # (3.18.4)
               Var_std_resid=np.var(e / np.sqrt(var + tau2)))                              # (3.17.1)
    if th_true is not None:
        out.update(RMSE_truth=np.sqrt(np.mean((pred - th_true) ** 2)),
                   Cov90_truth=np.mean(np.abs(pred - th_true) <= Z90 * np.sqrt(var)))      # (3.18.5)
    return out


def loo_or_block(th, tau2, X, F, fam, folds, use_delta, th_true=None):
    n = len(th); D = cdist(X, X); pred = np.zeros(n); var = np.zeros(n)
    for te in folds:
        tr = np.setdiff1d(np.arange(n), te); delta = tau2[tr] if use_delta else np.zeros(len(tr))
        par = np.exp(fit_reml(th[tr], F[tr], D[np.ix_(tr, tr)], delta, fam))
        m, v, _ = sk_predict(th[tr], F[tr], D[np.ix_(tr, tr)], delta, par, fam, X[tr], X[te], F[te], False)
        pred[te], var[te] = m, v
    return cv_metrics(th, pred, var, tau2, th_true), pred, var


In [ ]:
# =====================================================================================
# 9. Baseline demonstration + figures
# =====================================================================================
def decluster(y, u, dmin=3):
    """Cluster maxima of excesses, (3.3.1). y must be a regular daily array (NaN allowed)."""
    idx = np.where(y > u)[0]
    if len(idx) == 0:
        return np.array([])
    cl = [[idx[0]]]
    for i in idx[1:]:
        (cl[-1].append(i) if i - cl[-1][-1] <= dmin else cl.append([i]))
    return np.array([np.nanmax(y[c]) - u for c in cl])


def daily_demo(out, rng, plt, years=30):
    nd = 365 * years; t = np.arange(nd); z = np.zeros(nd); e = rng.standard_normal(nd)
    for k in range(1, nd):
        z[k] = 0.85 * z[k - 1] + np.sqrt(1 - .85 ** 2) * e[k]
    storm = np.where(rng.random(nd) < .03, stats.pareto.rvs(3.0, size=nd, random_state=rng), 1.0)
    Y = 5 * np.exp(.5 * z + .6 * np.sin(2 * np.pi * t / 365)) * storm
    us = np.quantile(Y, np.linspace(.85, .995, 40)); mrl = []; se = []; xis = []; sst = []; sxi = []; ssig = []
    for u in us:
        x = decluster(Y, u); N = len(x); mrl.append(x.mean()); se.append(1.96 * x.std(ddof=1) / np.sqrt(N))    # (3.5.3)
        ls, xi = fit_gpd(x); sg = np.exp(ls); C = gpd_cov(sg, xi, N)
        xis.append(xi); sxi.append(1.96 * np.sqrt(C[1, 1])); sst.append(sg - xi * u)
        ssig.append(1.96 * np.sqrt(C[0, 0] + u ** 2 * C[1, 1] - 2 * u * C[0, 1]))
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(us, mrl); ax[0].fill_between(us, np.array(mrl) - se, np.array(mrl) + se, alpha=.25)
    ax[0].set(title="Mean residual life (3.5.3)", xlabel="u", ylabel="e(u)")
    ax[1].errorbar(us, sst, ssig, fmt="o-", ms=3); ax[1].set(title="Modified scale  sigma*=sigma-xi u", xlabel="u")
    ax[2].errorbar(us, xis, sxi, fmt="o-", ms=3); ax[2].set(title="Shape xi stability", xlabel="u")
    u = np.quantile(Y, .97)
    for a_ in ax:
        a_.axvline(u, color="r", ls="--")
    fig.tight_layout(); fig.savefig(f"{out}/figures/fig_threshold_diagnostics.png", dpi=150); plt.close(fig)
    x = decluster(Y, u); N = len(x); ls, xi = fit_gpd(x); sg = np.exp(ls); lam = N / years
    xs = np.sort(x); p = (np.arange(1, N + 1) - .5) / N; Hx = 1 - (1 + xi * xs / sg) ** (-1 / xi)
    q = sg / xi * ((1 - p) ** (-xi) - 1)
    Ts = np.logspace(np.log10(max(2, 1.01 / lam)), 3, 60); zT = rl(sg, xi, lam, Ts, u)
    tauT = np.array([np.sqrt(_rl_logvar_u(sg, xi, lam, years, N, T_, u)) for T_ in Ts])
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    ax[0].plot(p, Hx, "o", ms=3); ax[0].plot([0, 1], [0, 1], "r"); ax[0].set(title="Probability plot", xlabel="empirical", ylabel="model")
    ax[1].plot(q, xs, "o", ms=3); ax[1].plot([0, xs.max()], [0, xs.max()], "r"); ax[1].set(title="Quantile plot", xlabel="model", ylabel="empirical")
    ax[2].semilogx(Ts, zT, "b"); ax[2].fill_between(Ts, zT * np.exp(-1.96 * tauT), zT * np.exp(1.96 * tauT), alpha=.25)
    ax[2].semilogx(years / (np.arange(1, N + 1) - .5), u + xs[::-1], "ko", ms=3)
    ax[2].set(title="Return-level plot (delta-method 95% band)", xlabel="T (years)", ylabel="discharge")
    fig.tight_layout(); fig.savefig(f"{out}/figures/fig_site_diagnostics.png", dpi=150); plt.close(fig)
    # GEV <-> GPD link (3.8.3)
    Tl = np.array([50., 100., 500.]); mu = u + sg * (lam ** xi - 1) / xi; sG = sg * lam ** xi
    zg = mu - sG / xi * (1 - (-np.log(1 - 1 / Tl)) ** (-xi))
    link = pd.DataFrame(dict(T=Tl, z_GPD=rl(sg, xi, lam, Tl, u), z_GEV_link=zg))
    link.to_csv(f"{out}/tables/GPD_GEV_link_check.csv", index=False)
    say(f"\n[daily demo] u(97th pct)={u:.2f}, clusters={N}, xi_hat={xi:.3f}, sigma_hat={sg:.2f}; "
        f"GPD vs GEV-link return levels:\n{link.round(3).to_string(index=False)}")


def _rl_logvar_u(sig, xi, lam, ny, N, T, u):
    xi = float(np.clip(xi, -.4, .9)); xs = xi if abs(xi) > 1e-4 else 1e-4; a = (lam * T) ** xs
    g = np.array([(a - 1) / xs, sig / xs * (a * np.log(lam * T) - (a - 1) / xs)])
    return (g @ gpd_cov(sig, max(xi, -0.25), N) @ g + (sig * a / lam) ** 2 * lam / ny) / rl(sig, xi, lam, T, u) ** 2


def hand_field(X):
    """Synthetic height-above-nearest-drainage surface around a meandering river (flood-extent demo)."""
    riv = 50 + 15 * np.sin(2 * np.pi * X[:, 0] / 100)
    return 8 * (1 - np.exp(-np.abs(X[:, 1] - riv) / 25)), riv


def stage_demo(out, seed, plt, jobs):
    rng = np.random.default_rng(seed); cfg = dict(CFG); fam = cfg["fam_fit"]
    daily_demo(out, rng, plt)
    data = simulate(rng, SCENARIOS["S1 Baseline"], 100, 200)
    Xg = data["Xg"]; n = len(Xg); Fg = design(Xg); D = cdist(Xg, Xg); T = 100
    a = atsite(data, rng, cfg, True); th, tau2 = a["th"], a["tau2"]
    # ---- fig: network, covariates, tau
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
    sc_ = ax[0].scatter(Xg[:, 0], Xg[:, 1], c=data["th_g"], s=data["ny"] * 3, cmap="viridis"); plt.colorbar(sc_, ax=ax[0])
    ax[0].set(title="Gauges (size = record length, colour = true ln z100)")
    ax[1].scatter(data["th_g"], th, c=np.sqrt(tau2)); ax[1].plot(*[[data["th_g"].min(), data["th_g"].max()]] * 2, "r")
    ax[1].set(title="At-site ln z100 vs truth (colour = tau_i)", xlabel="true", ylabel="MLE")
    ax[2].scatter(data["ny"], np.sqrt(tau2)); ax[2].set(title="Site noise tau_i vs record length", xlabel="years", ylabel="tau_i")
    fig.tight_layout(); fig.savefig(f"{out}/figures/fig_network_and_noise.png", dpi=150); plt.close(fig)
    # ---- Fig 4.6 posterior
    i_s, i_l = int(np.argmin(data["ny"])), int(np.argmax(data["ny"]))
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for a_, i, lab in zip(ax, (i_s, i_l), ("shortest", "longest")):
        d = a["draws"][i]; a_.scatter(d["ls"], d["xi"], s=6, alpha=.4); a_.plot(*d["mle"], "r*", ms=12)
        a_.set(title=f"Posterior (ln sigma, xi): {lab} record ({data['ny'][i]} yr)", xlabel="ln sigma", ylabel="xi")
    fig.tight_layout(); fig.savefig(f"{out}/figures/fig4_6_posterior_sigma_xi.png", dpi=150); plt.close(fig)
    # ---- REML OK vs SK: spatial variance (3.19) + variogram
    z0 = np.zeros(n); pOK = np.exp(fit_reml(th, Fg, D, z0, fam)); pSK = np.exp(fit_reml(th, Fg, D, tau2, fam))
    cmp = pd.DataFrame([dict(model="OK", sigma2_M=pOK[0], range_phi=pOK[1], nugget=pOK[2]),
                        dict(model="SK", sigma2_M=pSK[0], range_phi=pSK[1], nugget=pSK[2])])
    cmp.to_csv(f"{out}/tables/OK_vs_SK_covariance_parameters.csv", index=False)
    say("\n[demo] REML covariance parameters (thesis reports sigma2_M 0.717 for OK vs 0.041 for SK):\n"
        + cmp.round(4).to_string(index=False))
    r_ = th - Fg @ np.linalg.lstsq(Fg, th, rcond=None)[0]; iu = np.triu_indices(n, 1)
    h = D[iu]; g = .5 * (r_[:, None] - r_[None, :])[iu] ** 2; bins = np.linspace(0, 70, 12); bc = .5 * (bins[1:] + bins[:-1])
    gb = [g[(h >= bins[k]) & (h < bins[k + 1])].mean() for k in range(len(bc))]; hh = np.linspace(0, 70, 100)
    fig, ax = plt.subplots(figsize=(6, 4)); ax.plot(bc, gb, "ko", label="empirical (trend residuals)")
    for p_, lab in ((pOK, "OK"), (pSK, "SK")):
        ax.plot(hh, p_[2] + p_[0] * (1 - cov_fun(hh, 1, p_[1], fam)), label=lab)               # (3.10.8)
    ax.set(xlabel="h", ylabel="semivariance", title="Semivariogram"); ax.legend()
    fig.savefig(f"{out}/figures/fig_variogram.png", dpi=150); plt.close(fig)
    # ---- LOOCV and block CV
    folds = [[i] for i in range(n)]; blocks = np.array_split(np.argsort(Xg[:, 0]), 5)
    rows = []
    for nm, ud in (("OK", False), ("EVT-SK (direct)", True)):
        for cvn, fl in (("LOOCV", folds), ("Block5", blocks)):
            m, pr, vr = loo_or_block(th, tau2, Xg, Fg, fam, fl, ud, data["th_g"])
            rows.append(dict(method=nm, cv=cvn, **m))
            if nm == "EVT-SK (direct)" and cvn == "LOOCV":
                loo_pr = pr
    cvdf = pd.DataFrame(rows); cvdf.to_csv(f"{out}/tables/cross_validation_metrics.csv", index=False)
    say("\n[demo] Cross-validation (vs at-site estimates and, *_truth, vs simulated truth):\n"
        + cvdf.round(3).to_string(index=False))
    fig, ax = plt.subplots(figsize=(4.8, 4.5)); ax.scatter(th, loo_pr, c=np.sqrt(tau2)); ax.plot(*[[th.min(), th.max()]] * 2, "r")
    ax.set(xlabel="at-site ln z100", ylabel="LOOCV prediction", title="EVT-SK LOOCV"); fig.savefig(f"{out}/figures/fig_loocv.png", dpi=150); plt.close(fig)
    # ---- calibration on 200 validation sites
    pred = predict_methods(data, rng, cfg, data["Xv"], ALL[2:] + [])
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2)); al = np.linspace(.02, .98, 49); sc_rows = []
    for m in ALL[2:]:
        s, pit = score(pred[m], data["th_v"]); sc_rows.append(dict(method=m, **s))
        ax[1].plot(1 - al, [np.mean((pit >= x / 2) & (pit <= 1 - x / 2)) for x in al], label=m)  # c(alpha) (3.18.9)
        ax[0].hist(pit, bins=10, range=(0, 1), histtype="step", density=True, label=m)
    ax[1].plot([0, 1], [0, 1], "k--"); ax[1].set(title="Coverage curve", xlabel="nominal", ylabel="empirical"); ax[1].legend(fontsize=7)
    ax[0].set(title="PIT histogram (validation sites)"); ax[0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(f"{out}/figures/fig_calibration.png", dpi=150); plt.close(fig)
    pd.DataFrame(sc_rows).to_csv(f"{out}/tables/baseline_validation_scores.csv", index=False)
    say("\n[demo] Baseline out-of-sample scores (200 validation sites):\n" + pd.DataFrame(sc_rows).round(3).to_string(index=False))
    # ---- predictive distribution at one site (Fig 4.10)
    zz = np.exp(pred["Bayes EVT-SK (composed)"]["draws"][:, 0]); fig, ax = plt.subplots(figsize=(5, 4))
    ax.hist(zz, bins=30); ax.axvline(np.exp(data["th_v"][0]), color="r"); ax.set(title="Bayes EVT-SK predictive z100 (red = truth)")
    fig.savefig(f"{out}/figures/fig4_10_predictive_density.png", dpi=150); plt.close(fig)
    grid_products(data, a, cfg, rng, out, plt)


def grid_products(data, a, cfg, rng, out, plt, B=120):
    """Maps on the 30x30 grid: return level, uncertainty split (3.11.8/3.14.4), CVaR (3.15), flood extent (3.12.3)."""
    g = np.linspace(0, 100, 30); GX, GY = np.meshgrid(g, g); X0 = np.c_[GX.ravel(), GY.ravel()]
    Xg = data["Xg"]; Fg = design(Xg); F0 = design(X0); D = cdist(Xg, Xg); T = data["T"]; fam = cfg["fam_fit"]
    m, v, at, _ = krige_cols(a["ing"], a["ingv"], Fg, D, Xg, X0, F0, fam, 1.0)
    z = Psi(m, T); G = Psi_grad(m, T)
    sd_site = np.sqrt((G ** 2 * at).sum(1)) / z; sd_sp = np.sqrt(np.maximum((G ** 2 * (v - at)).sum(1), 0)) / z
    dr = krige_cols_bayes(a["ing_b"], a["ingv_b"], Fg, D, Xg, X0, F0, fam, 1.0, B, rng)       # (B,900,3)
    zb = Psi(dr, T); xi_r = dr[..., 1]; sg = np.exp(dr[..., 0]); xc = np.clip(xi_r, -.4, .9)
    cvar = (zb + sg - xc * U0) / (1 - xc)                                                        # (3.15.2)
    ratio = np.median(cvar / zb, 0); pfin = (xi_r < 1).mean(0)                                   # (3.15.3)-(3.15.4)
    hand, riv = hand_field(X0); depth = 0.6 * zb ** 0.4                                          # G(.) : crude rating curve
    inund = depth > hand[None, :]; pin = inund.mean(0); area = inund.mean(1)
    lo, hi = np.quantile(area, [.05, .95])
    say(f"\n[demo] Flood extent (3.12.8): median inundated fraction {np.median(area):.3f}, 90% interval [{lo:.3f}, {hi:.3f}]"
        "  (NB: cells are drawn independently given eta, so the area interval is optimistic; use a joint spatial draw for publication)")
    pd.DataFrame(dict(x=X0[:, 0], y=X0[:, 1], z100_freq=z, z100_bayes_median=np.median(zb, 0),
                      z100_q05=np.quantile(zb, .05, 0), z100_q95=np.quantile(zb, .95, 0),
                      sd_log_spatial=sd_sp, sd_log_atsite=sd_site, cvar_ratio=ratio, prob_cvar_finite=pfin,
                      prob_inundated=pin)).to_csv(f"{out}/tables/grid_predictions.csv", index=False)
    panels = [("Bayes median z100", np.median(zb, 0)), ("90% interval width (Bayes)", np.quantile(zb, .95, 0) - np.quantile(zb, .05, 0)),
              ("sd(ln z): spatial interpolation", sd_sp), ("sd(ln z): propagated at-site", sd_site),
              ("CVaR / VaR ratio (Fig 4.11)", ratio), ("P(inundated)", pin)]
    fig, ax = plt.subplots(2, 3, figsize=(15, 9))
    for a_, (ttl, arr) in zip(ax.ravel(), panels):
        im = a_.imshow(arr.reshape(30, 30), origin="lower", extent=[0, 100, 0, 100], cmap="viridis"); plt.colorbar(im, ax=a_, shrink=.8)
        a_.plot(Xg[:, 0], Xg[:, 1], "w^", ms=4, mec="k"); a_.set_title(ttl)
    ax[1, 2].plot(g, 50 + 15 * np.sin(2 * np.pi * g / 100), "r-", lw=1)
    fig.tight_layout(); fig.savefig(f"{out}/figures/fig4_8_maps_uncertainty_cvar_extent.png", dpi=150); plt.close(fig)


# =====================================================================================
# 10. Monte-Carlo, stress and sensitivity stages
# =====================================================================================
def stage_mc(out, seed, R, jobs):
    tasks = [(nm, sc, r, np.random.SeedSequence([seed, k, r]), dict(CFG), ALL, "mc")
             for k, (nm, sc) in enumerate(SCENARIOS.items()) for r in range(R)]
    say(f"\n[mc] {len(SCENARIOS)} scenarios x R={R} replicates x {len(ALL)} methods; jobs={jobs}")
    df = run_tasks(tasks, jobs, "mc"); df.to_csv(f"{out}/tables/mc_raw_results.csv", index=False)
    return df


def summarise_mc(df, out, plt):
    for met, nm in (("rmse", "RMSE"), ("cov", "Coverage90"), ("crps", "CRPS"), ("width", "Width90")):
        t = df.pivot_table(index="scenario", columns="method", values=met, aggfunc="mean")[ALL]
        t.to_csv(f"{out}/tables/table4_7_{nm}.csv"); say(f"\n[mc] Mean {nm} by scenario:\n{t.round(3).to_string()}")
    fig, ax = plt.subplots(1, 2, figsize=(17, 5)); sc = list(df.scenario.unique()); w = .11
    for k, m in enumerate(ALL):
        s = df[df.method == m].groupby("scenario")
        ax[0].bar(np.arange(len(sc)) + k * w, s.rmse.mean().reindex(sc), w, yerr=s.rmse.std().reindex(sc) / np.sqrt(s.size().reindex(sc)), label=m)
        ax[1].bar(np.arange(len(sc)) + k * w, s["cov"].mean().reindex(sc), w, label=m)
    for a_ in ax:
        a_.set_xticks(np.arange(len(sc)) + 3 * w); a_.set_xticklabels([x.split(" ", 1)[0] for x in sc])
    ax[1].axhline(.9, color="k", ls="--"); ax[1].set_ylim(.4, 1.02)
    ax[0].set(title="Out-of-sample RMSE of ln z100 (truth-based)"); ax[1].set(title="Empirical coverage of nominal 90% intervals"); ax[0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(f"{out}/figures/fig_mc_rmse_coverage.png", dpi=150); plt.close(fig)


def stage_tests(df, out):
    h2, h3 = paired_tests(df)
    h2.to_csv(f"{out}/tables/H2_wilcoxon_holm_vs_EVT-NN.csv", index=False); h3.to_csv(f"{out}/tables/H3_wilcoxon_holm_all.csv", index=False)
    R = df.groupby("scenario").rep.nunique().min()
    say(f"\n[H2] EVT-SK vs EVT alone (EVT-NN): {int(h2.reject_H0.sum())}/{len(h2)} Holm-adjusted tests reject H0 (R={R} per scenario)")
    say(h2.groupby("proposed").agg(scen_sig=("reject_H0", "sum"), mean_reduction_pct=("reduction_pct", "mean"), mean_win=("win_rate", "mean")).round(3).to_string())
    say(f"\n[H3] EVT-SK vs all conventional methods: {int(h3.reject_H0.sum())}/{len(h3)} tests reject H0")
    say(h3.groupby(["proposed", "conventional"]).agg(scen_sig=("reject_H0", "sum"), mean_reduction_pct=("reduction_pct", "mean"), mean_win=("win_rate", "mean")).round(3).to_string())
    return h2, h3


def stage_stress(out, seed, R, jobs, plt):
    conds = {"Longest record length": ("ny", [(5, 80), (5, 60), (5, 40), (5, 20), (5, 10)], [80, 60, 40, 20, 10]),
             "Contaminated gauges": ("contam_g", [0, .05, .10, .15, .20], [0, 5, 10, 15, 20]),
             "Number of gauges": ("n", [10, 15, 20, 30, 40], [10, 15, 20, 30, 40]),
             "True tail index": ("xi", [.05, .15, .25, .35, .45], [.05, .15, .25, .35, .45])}
    tasks = []
    for ci, (cn, (key, vals, lab)) in enumerate(conds.items()):
        for li, v in enumerate(vals):
            for r in range(R):
                tasks.append((f"{cn}|{lab[li]}", {key: v}, r, np.random.SeedSequence([seed, 9, ci, li, r]), dict(CFG), ALL, "stress"))
    say(f"\n[stress] {len(tasks)} runs (R={R} per level)")
    df = run_tasks(tasks, jobs, "stress"); df.to_csv(f"{out}/tables/stress_raw_results.csv", index=False)
    df["condition"] = df.scenario.str.split("|").str[0]; df["level"] = df.scenario.str.split("|").str[1].astype(float)
    t = df.groupby(["condition", "level", "method"])[["rmse", "cov", "crps"]].mean().reset_index()
    t.to_csv(f"{out}/tables/stress_summary.csv", index=False)
    fig, ax = plt.subplots(1, 4, figsize=(20, 4))
    for a_, cn in zip(ax, conds):
        for m in ALL:
            s = t[(t.condition == cn) & (t.method == m)]; a_.plot(s.level, s.rmse, "o-", ms=3, label=m)
        a_.set_title(cn); a_.set_ylabel("RMSE")
    ax[0].legend(fontsize=6); fig.tight_layout(); fig.savefig(f"{out}/figures/fig_stress_tests.png", dpi=150); plt.close(fig)
    say(t.pivot_table(index=["condition", "level"], columns="method", values="rmse")[ALL].round(3).to_string())


def stage_sens(out, seed, R, jobs, plt):
    settings = ([("cov_family", f, dict(fam_fit=f)) for f in ("exp", "m32", "m52", "gauss")]
                + [("return_period", T, dict(T=T)) for T in (50, 100, 500)]
                + [("noise_scale_c", c, dict(noise_scale=c)) for c in (.25, 1, 2, 4)]
                + [("trend", tr, dict(use_trend=tr)) for tr in (True, False)])
    tasks = []
    for fam_, val, upd in settings:
        for r in range(R):                                    # same data across settings (paired)
            tasks.append((f"{fam_}|{val}", {}, r, np.random.SeedSequence([seed, 7, r]), {**CFG, **upd}, ALL[2:], "sens"))
    say(f"\n[sens] {len(tasks)} runs (R={R} per setting)")
    df = run_tasks(tasks, jobs, "sens"); df.to_csv(f"{out}/tables/sensitivity_raw_results.csv", index=False)
    df["factor"] = df.scenario.str.split("|").str[0]; df["value"] = df.scenario.str.split("|").str[1]
    t = df.groupby(["factor", "value", "method"])[["rmse", "cov", "crps", "width"]].mean().reset_index()
    t.to_csv(f"{out}/tables/sensitivity_summary.csv", index=False)
    say(t.pivot_table(index=["factor", "value"], columns="method", values="rmse")[ALL[2:]].round(3).to_string())
    say("\n[sens] 90% coverage:\n" + t.pivot_table(index=["factor", "value"], columns="method", values="cov")[ALL[2:]].round(3).to_string())
    fa = list(dict.fromkeys(t.factor)); fig, ax = plt.subplots(1, len(fa), figsize=(5 * len(fa), 4))
    for a_, f in zip(ax, fa):
        s = t[t.factor == f].pivot_table(index="value", columns="method", values="rmse")[ALL[2:]]
        s.plot(kind="bar", ax=a_, legend=False); a_.set_title(f"Sensitivity: {f}"); a_.set_ylabel("RMSE")
    ax[0].legend(fontsize=6); fig.tight_layout(); fig.savefig(f"{out}/figures/fig_sensitivity.png", dpi=150); plt.close(fig)


In [ ]:
# =====================================================================================
# 11. Real-data pipeline (Ghana basin application)
# =====================================================================================
def real_data(args):
    """daily CSV  : gauge_id,date,discharge          (one row per gauge-day; gaps allowed)
       gauges CSV : gauge_id,x,y,<covariates...>     (projected coordinates, e.g. km)
       targets CSV: id,x,y,<covariates...>           (ungauged prediction sites)
    Per gauge: QC -> threshold (quantile --q) -> declustering (--dmin days) -> GPD ML & Bayes -> tau_i^2.
    Spatial: SK on (ln sigma, xi, ln lambda, ln u) -> return level, 90% interval, CVaR, P(CVaR finite)."""
    out = args.out; os.makedirs(out, exist_ok=True); rng = np.random.default_rng(args.seed)
    daily = pd.read_csv(args.daily, parse_dates=["date"]); G = pd.read_csv(args.gauges); Tg = pd.read_csv(args.targets)
    covs = [c for c in args.covariates.split(",") if c]; T = args.T; exc = []; ny = []; us = []; ids = []
    for gid, g in daily.groupby("gauge_id"):
        y = g.set_index("date")["discharge"].where(lambda s: s >= 0).sort_index()
        y = y[~y.index.duplicated()].asfreq("D"); yrs = y.notna().sum() / 365.25
        if yrs < args.min_years or gid not in set(G.gauge_id):
            continue
        u = float(np.nanquantile(y.values, args.q)); x = decluster(y.values, u, args.dmin)
        if len(x) < 8:
            continue
        exc.append(x); ny.append(yrs); us.append(u); ids.append(gid)
    G = G.set_index("gauge_id").loc[ids].reset_index(); n = len(ids)
    say(f"[real] {n} gauges retained (>= {args.min_years} yr, >= 8 clusters)")
    mu_, sd_ = G[covs].mean(), G[covs].std().replace(0, 1)
    mk = lambda df_: np.c_[np.ones(len(df_)), ((df_[covs] - mu_) / sd_).values] if covs else np.ones((len(df_), 1))
    Xg = G[["x", "y"]].values; X0 = Tg[["x", "y"]].values; Fg, F0 = mk(G), mk(Tg); D = cdist(Xg, Xg)
    cfg = dict(CFG); data = dict(exc=exc, ny=np.array(ny), T=T); fam = args.fam
    # at-site (return level at gauge i is relative to its own threshold u_i)
    th = np.zeros(n); tau2 = np.zeros(n); ing = np.zeros((n, 4)); ingv = np.zeros((n, 4)); msig = np.mean([np.log(x.mean()) for x in exc])
    ingb = np.zeros((n, 4)); ingvb = np.zeros((n, 4))
    for i, x in enumerate(exc):
        N = len(x); ls, xi = fit_gpd(x); lam = N / ny[i]; xc = np.clip(xi, -.4, .9)
        th[i] = np.log(rl(np.exp(ls), xi, lam, T, us[i]))
        tau2[i] = min(boot_logvar(x, ny[i], rng, 100, T) if N < 20 else _rl_logvar_u(np.exp(ls), xi, lam, ny[i], N, T, us[i]), 2.0)
        ing[i] = [ls, xi, np.log(lam), np.log(us[i])]; ingv[i] = [2 * (1 + xc) ** 2 / N, (1 + xc) ** 2 / N, 1 / N, 1e-6]
        d = bayes_site(x, ny[i], rng, msig, 1.0, 300); ll = np.log(d["lam"])
        ingb[i] = [np.median(d["ls"]), np.median(d["xi"]), np.median(ll), np.log(us[i])]
        ingvb[i] = [d["ls"].var(ddof=1), d["xi"].var(ddof=1), ll.var(ddof=1), 1e-6]
    pd.DataFrame(dict(gauge_id=ids, years=ny, u=us, n_clusters=[len(x) for x in exc], ln_sigma=ing[:, 0], xi=ing[:, 1],
                      lam=np.exp(ing[:, 2]), ln_zT=th, tau=np.sqrt(tau2))).to_csv(f"{out}/real_at_site_estimates.csv", index=False)
    m, v, at, pars = krige_cols(ing, ingv, Fg, D, Xg, X0, F0, fam, 1.0)
    u0 = np.exp(m[:, 3]); z = Psi(m[:, :3], T, u0); Gd = Psi_grad(m[:, :3], T)
    dr = krige_cols_bayes(ingb, ingvb, Fg, D, Xg, X0, F0, fam, 1.0, args.B, rng)
    zb = Psi(dr[..., :3], T, np.exp(dr[..., 3])); xc = np.clip(dr[..., 1], -.4, .9)
    cv = (zb + np.exp(dr[..., 0]) - xc * np.exp(dr[..., 3])) / (1 - xc)
    res = Tg.copy()
    res["zT_freq"] = z; res["sd_log_spatial"] = np.sqrt(np.maximum((Gd ** 2 * (v[:, :3] - at[:, :3])).sum(1), 0)) / z
    res["sd_log_atsite"] = np.sqrt((Gd ** 2 * at[:, :3]).sum(1)) / z
    res["zT_bayes_median"] = np.median(zb, 0); res["zT_q05"], res["zT_q95"] = np.quantile(zb, [.05, .95], 0)
    res["CVaR_median"] = np.median(cv, 0); res["P_CVaR_finite"] = (dr[..., 1] < 1).mean(0)
    res.to_csv(f"{out}/real_predictions_T{int(T)}.csv", index=False)
    cvm, _, _ = loo_or_block(th, tau2, Xg, Fg, fam, [[i] for i in range(n)], True)
    say("[real] LOOCV (direct form):", {k: round(float(x), 3) for k, x in cvm.items()})
    say(f"[real] predictions written to {out}/real_predictions_T{int(T)}.csv")


In [ ]:
# =====================================================================================
# 12. Main
# =====================================================================================
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--out", default="results"); ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--R", type=int, default=250, help="Monte-Carlo replicates per scenario (max 250 requested)")
    ap.add_argument("--stress_R", type=int, default=15); ap.add_argument("--sens_R", type=int, default=30)
    ap.add_argument("--gof_B", type=int, default=199); ap.add_argument("--jobs", type=int, default=max(1, (os.cpu_count() or 2) - 1))
    ap.add_argument("--stages", default="demo,gof,mc,stress,sens,tests"); ap.add_argument("--quick", action="store_true")
    ap.add_argument("--real", action="store_true"); ap.add_argument("--daily"); ap.add_argument("--gauges"); ap.add_argument("--targets")
    ap.add_argument("--covariates", default=""); ap.add_argument("--q", type=float, default=.97); ap.add_argument("--dmin", type=int, default=3)
    ap.add_argument("--min_years", type=float, default=10); ap.add_argument("--T", type=float, default=100); ap.add_argument("--fam", default="m32")
    ap.add_argument("--B", type=int, default=200)

    # Use parse_known_args() to ignore unrecognized Jupyter kernel configuration flags
    a, _ = ap.parse_known_args()

    if a.real:
        return real_data(a)
    if a.quick:
        a.R, a.stress_R, a.sens_R, a.gof_B = 4, 2, 2, 29
    a.R = min(a.R, 250)
    os.makedirs(f"{a.out}/tables", exist_ok=True); os.makedirs(f"{a.out}/figures", exist_ok=True)
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    st = set(a.stages.split(",")); t0 = time.time(); df = None
    say(f"EVT-SK thesis pipeline | R={a.R} stress_R={a.stress_R} sens_R={a.sens_R} jobs={a.jobs} seed={a.seed}")
    if "demo" in st:
        stage_demo(a.out, a.seed, plt, a.jobs)
    if "gof" in st:
        stage_gof(a.out, a.seed + 1, a.gof_B)
    if "mc" in st:
        df = stage_mc(a.out, a.seed, a.R, a.jobs); summarise_mc(df, a.out, plt)
    if "stress" in st:
        stage_stress(a.out, a.seed, a.stress_R, a.jobs, plt)
    if "sens" in st:
        stage_sens(a.out, a.seed, a.sens_R, a.jobs, plt)
    if "tests" in st:
        if df is None:
            df = pd.read_csv(f"{a.out}/tables/mc_raw_results.csv")
        stage_tests(df, a.out)
    say(f"\nFinished in {(time.time() - t0) / 60:.1f} min. Outputs in ./{a.out}/tables and ./{a.out}/figures")
    open(f"{a.out}/summary_log.txt", "w").write("\n".join(LOG))


if __name__ == "__main__":
    main()

EVT-SK thesis pipeline | R=250 stress_R=15 sens_R=30 jobs=1 seed=2026

[daily demo] u(97th pct)=17.07, clusters=113, xi_hat=0.297, sigma_hat=5.31; GPD vs GEV-link return levels:
    T   z_GPD  z_GEV_link
 50.0  83.876      83.623
100.0 103.237     103.082
500.0 167.018     166.968

[demo] REML covariance parameters (thesis reports sigma2_M 0.717 for OK vs 0.041 for SK):
model  sigma2_M  range_phi  nugget
   OK   18.2096    316.882  0.0583
   SK    2.9260    120.471  0.0237

[demo] Cross-validation (vs at-site estimates and, *_truth, vs simulated truth):
         method     cv  RMSE   MAE   NSE  Coverage90  Var_std_resid  RMSE_truth  Cov90_truth
             OK  LOOCV 0.309 0.244 0.783       0.800          1.762       0.218        0.667
             OK Block5 0.372 0.286 0.686       0.800          1.320       0.217        0.900
EVT-SK (direct)  LOOCV 0.341 0.272 0.736       0.767          2.009       0.270        0.600
EVT-SK (direct) Block5 0.382 0.298 0.668       0.767          1.417 

In [ ]:
import pandas as pd
import numpy as np

# Initialize generator with the same seed used in main()
rng = np.random.default_rng(2026)
cfg = dict(CFG)

# Simulate baseline dataset (used in the stage_demo function)
data = simulate(rng, SCENARIOS["S1 Baseline"], 100, 200)
Xg = data["Xg"]

# Fit at-site estimates to obtain estimated return levels (th) and estimation variances (tau2)
a = atsite(data, rng, cfg, bayes=True)
th = a["th"]
tau2 = a["tau2"]

# Create a DataFrame containing all variables plotted in fig_network_and_noise.png
network_noise_df = pd.DataFrame({
    "gauge_index": np.arange(len(Xg)) + 1,
    "coord_x": Xg[:, 0],
    "coord_y": Xg[:, 1],
    "record_length_years": data["ny"],
    "true_ln_z100": data["th_g"],
    "estimated_ln_z100_mle": th,
    "site_noise_tau_i": np.sqrt(tau2)
})

display(network_noise_df)

,gauge_index,coord_x,coord_y,record_length_years,true_ln_z100,estimated_ln_z100_mle,site_noise_tau_i
0,1,63.991317,46.726840,17,4.035573,3.973966,0.201579
1,2,37.050053,35.491733,18,4.124865,4.010964,0.201165
2,3,79.051825,90.514384,40,3.887627,3.635983,0.117784
3,4,17.735319,65.278480,33,3.407813,3.443725,0.210762
4,5,29.830277,96.696220,26,3.270660,3.072366,0.057119
5,6,91.985016,63.587080,37,4.144123,4.115530,0.176188
6,7,75.273210,51.515370,24,4.040211,3.631885,0.066575
7,8,82.589526,44.838055,16,4.338133,4.588251,0.517534
8,9,33.881245,27.789922,36,4.305923,3.901914,0.136849
9,10,22.633306,52.581684,21,3.659222,3.797802,0.230176


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Recreate the synthetic daily data using the original seed from daily_demo
rng = np.random.default_rng(2026)  # seed used in daily_demo
years = 30
nd = 365 * years
t = np.arange(nd)
z = np.zeros(nd)
e = rng.standard_normal(nd)
for k in range(1, nd):
    z[k] = 0.85 * z[k - 1] + np.sqrt(1 - .85 ** 2) * e[k]

storm = np.where(rng.random(nd) < .03, stats.pareto.rvs(3.0, size=nd, random_state=rng), 1.0)
Y = 5 * np.exp(.5 * z + .6 * np.sin(2 * np.pi * t / 365)) * storm

# Replicate threshold diagnostics data generation
us = np.quantile(Y, np.linspace(.85, .995, 40))
rows = []

for u in us:
    x = decluster(Y, u)
    N = len(x)
    mrl = x.mean()
    mrl_se = 1.96 * x.std(ddof=1) / np.sqrt(N)

    ls, xi = fit_gpd(x)
    sg = np.exp(ls)
    C = gpd_cov(sg, xi, N)

    # modified scale sigma* = sigma_u - xi * u
    mod_scale = sg - xi * u
    mod_scale_se = 1.96 * np.sqrt(C[0, 0] + u ** 2 * C[1, 1] - 2 * u * C[0, 1])

    shape_se = 1.96 * np.sqrt(C[1, 1])

    rows.append({
        "threshold_u": u,
        "clusters_N": N,
        "mean_residual_life": mrl,
        "mrl_95_ci_half_width": mrl_se,
        "modified_scale": mod_scale,
        "modified_scale_95_ci_half_width": mod_scale_se,
        "shape_xi": xi,
        "shape_xi_95_ci_half_width": shape_se
    })

threshold_diagnostics_df = pd.DataFrame(rows)
display(threshold_diagnostics_df)

,threshold_u,clusters_N,mean_residual_life,mrl_95_ci_half_width,modified_scale,modified_scale_95_ci_half_width,shape_xi,shape_xi_95_ci_half_width
0,10.105939,295,6.481125,0.972969,2.735938,2.145550,0.225849,0.139889
1,10.193532,293,6.472276,0.975358,2.750530,2.161636,0.223205,0.140062
2,10.330066,296,6.367838,0.965795,2.362122,2.163015,0.240295,0.141298
3,10.466252,293,6.329045,0.971643,2.112215,2.192945,0.251882,0.143346
4,10.601177,289,6.315188,0.979195,2.025504,2.225596,0.254407,0.144626
5,10.703301,288,6.325181,0.978556,2.105350,2.241457,0.248446,0.144188
6,10.809267,280,6.398322,0.998634,2.257783,2.293106,0.241016,0.145363
7,10.907700,282,6.322735,0.989928,2.065163,2.291980,0.247607,0.145616
8,11.045027,273,6.406857,1.012227,2.300442,2.352206,0.235371,0.146545
9,11.165248,269,6.389827,1.022335,2.140754,2.387958,0.242281,0.148457


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 250, 'savefig.dpi': 250, 'font.size': 10})

# 1. Redraw Threshold Diagnostics Plot with even higher resolution (250 DPI)
us = threshold_diagnostics_df['threshold_u'].values
mrl = threshold_diagnostics_df['mean_residual_life'].values
se = threshold_diagnostics_df['mrl_95_ci_half_width'].values
sst = threshold_diagnostics_df['modified_scale'].values
ssig = threshold_diagnostics_df['modified_scale_95_ci_half_width'].values
xis = threshold_diagnostics_df['shape_xi'].values
sxi = threshold_diagnostics_df['shape_xi_95_ci_half_width'].values

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
ax[0].plot(us, mrl, color='tab:blue', lw=2)
ax[0].fill_between(us, mrl - se, mrl + se, alpha=0.2, color='tab:blue')
ax[0].set_title("Mean Residual Life (3.5.3)", fontsize=11, fontweight='bold')
ax[0].set_xlabel("Threshold u")
ax[0].set_ylabel("Mean Excess e(u)")

ax[1].errorbar(us, sst, yerr=ssig, fmt="o-", ms=4, color='tab:orange', ecolor='gray', elinewidth=1.5, capsize=2, label="sigma*")
ax[1].set_title("Modified Scale sigma* = sigma - xi * u", fontsize=11, fontweight='bold')
ax[1].set_xlabel("Threshold u")

ax[2].errorbar(us, xis, yerr=sxi, fmt="o-", ms=4, color='tab:green', ecolor='gray', elinewidth=1.5, capsize=2, label="xi")
ax[2].set_title("Shape Parameter xi Stability", fontsize=11, fontweight='bold')
ax[2].set_xlabel("Threshold u")

for a_ in ax:
    a_.axvline(x=27.7, color="tab:red", ls="--", lw=1.5, label="97th pct Threshold")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 200, 'savefig.dpi': 200, 'font.size': 10})

us = threshold_diagnostics_df['threshold_u'].values
mrl = threshold_diagnostics_df['mean_residual_life'].values
se = threshold_diagnostics_df['mrl_95_ci_half_width'].values
sst = threshold_diagnostics_df['modified_scale'].values
ssig = threshold_diagnostics_df['modified_scale_95_ci_half_width'].values
xis = threshold_diagnostics_df['shape_xi'].values
sxi = threshold_diagnostics_df['shape_xi_95_ci_half_width'].values

fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
ax[0].plot(us, mrl, color='tab:blue', lw=2)
ax[0].fill_between(us, mrl - se, mrl + se, alpha=0.2, color='tab:blue')
ax[0].set_title("Mean Residual Life (3.5.3)", fontsize=11, fontweight='bold')
ax[0].set_xlabel("Threshold u")
ax[0].set_ylabel("Mean Excess e(u)")

ax[1].errorbar(us, sst, yerr=ssig, fmt="o-", ms=4, color='tab:orange', ecolor='gray', elinewidth=1.5, capsize=2)
ax[1].set_title("Modified Scale sigma* = sigma - xi * u", fontsize=11, fontweight='bold')
ax[1].set_xlabel("Threshold u")

ax[2].errorbar(us, xis, yerr=sxi, fmt="o-", ms=4, color='tab:green', ecolor='gray', elinewidth=1.5, capsize=2)
ax[2].set_title("Shape Parameter xi Stability", fontsize=11, fontweight='bold')
ax[2].set_xlabel("Threshold u")

for a_ in ax:
    a_.axvline(x=27.7, color="tab:red", ls="--", lw=1.5, label="97th pct Threshold")

plt.tight_layout()
plt.show()